In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn import model_selection, metrics

from mics import qrisk3, simulation

In [ ]:
# start small for a fast iteration, the scale up to 100k patients once the pipeline runs cleanly
n_patients = 1000 
random_seed = 42

In [ ]:
# Generate population
patients = simulation.generate_patients(n_patients, random_seed=random_seed)
display(patients)

# 

In [ ]:
# Prevelance of Type 2 diabetes in patients over 60 should be ~ 12%, rising with BMI 
patients.filter(
    pl.col("Age") > 60
)['Type 2 Diabetes'].value_counts(sort=True, normalize=True)

In [ ]:
# Type 2 diabetes prevalence should rise with age, and be higher in patients with higher BMI 
# Simple plot demonstrates this relationship - future step is to confrim via anaysis of the data
patients.group_by("Age").agg(
    (pl.col("Type 2 Diabetes")==1).mean().alias("Type 2 Diabetes")
).sort("Age").plot.line(x="Age", y="Type 2 Diabetes")

# Generate the 10 year risk for our patients

In [ ]:
# This is the first step to generating the 'y' variable in our formulation. 
# Now, we have X (a matrix of risk factors), and each patient's 10 year CVD risk using QRISK3
# This returns a percentage between 0 and 100 - the the probability of a patient having a CVD event in the next 10 years

# Need to confirm that the distribution of risk looks reasonable, 
# and that the relationship between risk and 'y' (CVD event) is as expected 
patient_risk = qrisk3.calculate_qrisk3(patients)

# Perform a sanity check before plotting and generating y 
print("Population Statistics:", patient_risk.describe())
 
# Convert risk scores to numpy array for plotting, outcome sampling and further analysis
risk_pct = patient_risk.to_numpy()

# Plot distribution of risk - should be right skewed 
#TODO: need to fix the x-axis to be between 0 and 100, and add a vertical line at the mean risk
sns.histplot(risk_pct, bins=50, kde=True)

# Display risk as percentages (0-100)
print("Patients risk as percentages (0-100):", risk_pct)

# Simulate the outcomes

In [ ]:
# Bernoulli sampling: 'y' is 1 if a CVD event occurs, otherwise 0 - binary outcome.
# To be used as the training target for model 1. Can the model learn the relationship between 'X' (risk factors), 
# and risk 'y' and therefore learn to predict 'y' from 'X'?

y = simulation.risk_to_event(risk_pct, random_seed=random_seed)

risk_prob = risk_pct / 100 # Recomputed for inspection

# Sanity check: 
# The event rate should be close to the mean calculated risk by QRISK3 
# The mean(y) should be close to the mean(risk_prob) - it checks that the Bernoulli sampling produces the correct ouput
print("Patients risk as predicted probabilities (0-1):", risk_prob[:10])
print("Simulated CVD events (y):", y[:10])

# Show comparisons 
print("Mean risk (%):", risk_pct.mean())
print("Event rate (%):", y.mean() * 100)
print("Event count:", y.sum())


In [ ]:
# Test the function on a larger sample of patients - n = 10,000

# Sanity check: 
# When scaled the function should give the same event rate as the mean risk percentage from QRISK3.
# The random variation averages out and mean(y) should be very closly matched to mean(risk_prob.
# This connfirms the Bernoulli sompling is producing the correct distribution at scale. 

# Generate larger population and convert into numpy array for analysis
large_patients = simulation.generate_patients(n=10000, random_seed=42)
large_risk_pct= qrisk3.calculate_qrisk3(large_patients).to_numpy()

# Simulate outcomes 
y_large = simulation.risk_to_event(large_risk_pct, random_seed=random_seed)

# Show comparisons
print("Mean risk (%):", large_risk_pct.mean())
print("Event rate (%):", y_large.mean() * 100)
print("Event count:", y_large.sum())

# Build the feature matrix 

In [ ]:
# Make a copy of the patients dataframe to avoid modifications to the original
matrix = patients.clone()

# One-hot encode the categorical variables 
# It turns each category into its own separate binary column (0/1)
matrix = matrix.to_dummies(columns=["Sex", "Ethnicity", "Smoking Status"])

# Convert into numpy array for model fitting
X = matrix.to_numpy()

# Sanity check:
# The shape of X should be (n_patients, n_features) - n_features is the number of risk factors after one-hot encoding
print("Column names:", matrix.columns)
print("Shape of X:", X.shape)
print("First row of X:", X[0])


# Fit a model to predict the risk

We want $y'=f(X)$ to estimate y

TODO: We don't have a test dataset here. We've used the whole `X` and `y` data to fit the model, so we can't evaluate how well it performs on unseen data. We should split our data into a training set and a test set to properly evaluate the model's performance.

In [ ]:
# Call fit_model function

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=random_seed)

model_1 = simulation.fit_model(X_train,y_train)

# Sanity check:
# The predicted event rate should be close to the actual event rate in 'y'; this ensures the model is well calibrated

print("Actual event rate:", y_test.mean())
print("Model 1 predicted event rate:", model_1.predict_proba(X_test)[:,1].mean()) 
# The predicted probabilities of a CVD event outcome for each patient
# It gives a two column array: the probability of the patient belonging to class 0 (no event) and class 1 (event). 
# We only care about the probability of an event - the second column; index 1.


#TODO: Inspect model coefficients to confirm that the model has learned the expected relationships between risk factors and CVD events.
# Does it make biological sense? 
# E.g. Are aga and AF positively associated with CVD risk? Is sex_female negatively associated with CVD risk? 
# Are the relationships in line with existing medical knowledge and literature on CVD risk factors?



## Analyse how good the model is at predicting the risk

### Confusion Matrix - but you need to choose a threshold

This is a bit arbitrary, and it doesn't capture the full picture of the model's performance across all possible thresholds.

In [ ]:
y_pred_prob = model_1.predict_proba(X_test)[:,1]
threshold = 0.5
metrics.ConfusionMatrixDisplay(
    metrics.confusion_matrix(
        y_true = y_test,
        y_pred = (y_pred_prob > threshold).astype(int)
    )
).plot()


### AUC 

This compares performance across all possible thresholds, and is a good overall measure of how well the model can discriminate between patients who will have an event and those who won't.

In [ ]:
metrics.RocCurveDisplay.from_predictions(y_test, y_pred_prob)
print("AUC-ROC:", metrics.roc_auc_score(y_test, y_pred_prob))

Violinplot shows the difference in the _scores_ between patients who had an event and those who didn't die. We expect to see higher predicted probabilities for patients who had an event (y=1) compared to those who didn't (y=0). If the model is performing well, the distribution of predicted probabilities for y=1 should be skewed towards higher values, while the distribution for y=0 should be skewed towards lower values.

In [ ]:
sns.violinplot(x=y_test, y=y_pred_prob, hue=y_test)
plt.xlabel("Actual CVD Event (y)")
plt.ylabel("Predicted CVD Risk Probability")
plt.ylim(0,1)

Conclusion: Model 1 is OK, possibly could be a bit better. 

Improvements might involve:
* Using a bigger dataset. 
* Using a more complex model (e.g. random forest, gradient boosting, neural network).

# Now, let's implement the model in a new population

In [ ]:
#
# Generate a new population. Use a different random seed so we don't get the exact same patients. 
X_2 = simulation.generate_patients(n_patients, random_seed=random_seed+1)
# Work out what model 1 thinks will happen. 
modelled_risk_2 = model_1.predict_proba(X_2)[:, 1]
# Imagine we're a GP, and we decide to put patients on statins if their predicted risk is above 10%.
on_statins = modelled_risk_2 > 0.1
# Add in a column to the dataframe to indicate whether patients are on statins or not.
X_2['on_statins'] = on_statins
# This bit may go wrong - need to make sure the calculate_qrisk3 function can handle the new column we've added in.
risk_2 = qrisk3.calculate_qrisk3(X_2)
# Now we can simulate the outcomes for this new population, and see how the model performs.
y_2 = np.random.uniform(0, 1, n_patients) < risk_2

# Now we train a second model on our second population. 

Problem: training a model $f_2(x)$ but not accounting for statins
This is the model-induced concept shift problem. 

In [ ]:
# MAKE SURE THAT MODEL 2 DOESN'T USE THE 'on_statins' COLUMN - IT SHOULDN'T BE A RISK FACTOR IN THE MODEL, IT'S AN INTERVENTION
# We can add this later, but as a first pass, we want the model to be unaware of the intervention, 
# and just see how the coefficients change after the intervention.
model_2 = simulation.fit_model(X_2, y_2)

# TODO: Compare the coefficients of model 1 and model 2 to see how they differ after the simulated intervention (statin prescription).
# Is there signs of coeffecient attenuation for the high-weight risk factors
# Are M2's coefficents smaller in magnitude than M1's - sign of MICS occuring (possible weaker risk-factor relationships due to the intervention)

# Generated comparative third population 

In [ ]:
# M2 should underpredict risk compared to M1 and the true event rate